# Intermediate SQL for Data Analysts
**DataraFlow Cohort 2 · Week 6 · Training notebook & reference**

This notebook takes you from *"I can write a SELECT"* to *"I can answer real business questions with SQL."* Every query runs against the live cohort database (`assessmentDB` on Azure SQL / Microsoft SQL Server), using the same tables you use in your weekly work.

### How to use it
1. Run the **Setup cells** first (Section 0).
2. Work through the sections in order. Read the explanation, run the example, then change it and run it again. Breaking a query on purpose is the fastest way to learn.
3. Try each **Exercise** before opening its solution.
4. Keep the notebook as a reference. **Section 14** is a one-page cheat sheet.

---

### Contents
| # | Section | You will learn |
|---|---|---|
| **0** | **Setup & ground rules** | Connecting, the read-only account, the `q()` helper |
| **1** | **Exploring an unfamiliar database** | `INFORMATION_SCHEMA`, profiling a table |
| **2** | **Refresher: the core SELECT** | Aliases, `TOP`, `DISTINCT`, bracket-quoting, `ORDER BY` |
| **3** | **Filtering like a pro** | `IN`, `BETWEEN`, `LIKE`, `NULL` traps, `COALESCE`, `NULLIF` |
| **4** | **Types, conversion & built-in functions** | `CAST`/`TRY_CONVERT`, dates, strings, maths, integer-division trap |
| **5** | **CASE expressions** | Bucketing, flags, conditional aggregation |
| **6** | **Aggregation in depth** | `GROUP BY`, `HAVING`, `COUNT` variants, `ROLLUP`, `STRING_AGG` |
| **7** | **Joins** | Inner, left, anti-join, full outer, self-join, row explosion |
| **8** | **Set operators** | `UNION` vs `UNION ALL`, `INTERSECT`, `EXCEPT` |
| **9** | **Subqueries** | Scalar, `IN`, `EXISTS`, correlated, derived tables |
| **10** | **Common Table Expressions (CTEs)** | Readable multi-step queries, recursive CTEs |
| **11** | **Window functions** | Ranking, top-N per group, running totals, moving averages, `LAG`/`LEAD` |
| **12** | **Reshaping** | `PIVOT`, `UNPIVOT` with `CROSS APPLY` |
| **13** | **SQL + Python like a professional** | Parameterised queries, good habits, performance |
| **14** | **Capstone project & cheat sheet** | An end-to-end analysis + quick reference |

> **Dialect note:** Our server is **Microsoft SQL Server (T-SQL)**. Roughly 90% of what you learn here works in PostgreSQL, MySQL, BigQuery and Snowflake too. Where T-SQL differs (`TOP` vs `LIMIT`, `[brackets]` vs `"quotes"`), the cheat sheet in Section 14 calls it out.


---
## 0. Setup & ground rules

### 0.1 Before you start
- You need the project's `.env` file with the read-only credentials (`SQL_READONLY_USERNAME` / `SQL_READONLY_PASSWORD`).
- Azure SQL only accepts connections from approved IP addresses. If you get an error like *"Client with IP address 'x.x.x.x' is not allowed to access the server"*, send that IP to the admin to add to the firewall.
- Use the project's virtual environment as your notebook kernel.

### 0.2 Read-only by design
Your account belongs to the `db_datareader` role. You can read every table, but you cannot `INSERT`, `UPDATE`, `DELETE`, `DROP`, `CREATE` or `ALTER` anything. The server enforces this. On top of that, the `q()` helper below refuses to send anything other than a read query, so a typo can't cause harm.

This is how most analysts work in industry: you get read access to production or warehouse data, and you create new results in your query or in pandas, never by changing the source tables.


In [19]:
# --- Connect to the cohort database (read-only) ---
import sys, pathlib, re
_p = pathlib.Path.cwd()
while _p != _p.parent and not (_p / "db" / "dataraflow_db.py").exists():
    _p = _p.parent
sys.path.insert(0, str(_p))

from db.dataraflow_db import run_query, get_engine, get_connection
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

_BLOCKED = re.compile(
    r"\b(INSERT|UPDATE|DELETE|MERGE|DROP|ALTER|CREATE|TRUNCATE|EXEC|EXECUTE|GRANT|REVOKE|DENY|INTO|BACKUP|RESTORE)\b",
    re.IGNORECASE,
)

def q(sql: str) -> pd.DataFrame:
    """Run a read-only query and return a DataFrame.
    String literals and comments are stripped before the safety check,
    so a value like 'Delete me' inside quotes is fine.
    """
    scrubbed = re.sub(r"'([^']|'')*'", "''", sql)          # string literals
    scrubbed = re.sub(r"--[^\n]*", "", scrubbed)             # -- comments
    scrubbed = re.sub(r"/\*.*?\*/", "", scrubbed, flags=re.S) # /* comments */
    hit = _BLOCKED.search(scrubbed)
    if hit:
        raise PermissionError(
            f"'{hit.group(0).upper()}' is not allowed: this notebook is read-only. "
            "Shape your data inside the SELECT, or in pandas."
        )
    return run_query(sql)

# Verify connection
q("SELECT DB_NAME() AS database_name, CURRENT_USER AS logged_in_as, SYSDATETIME() AS server_time")


,database_name,logged_in_as,server_time
0,assessmentDB,intern_readonly,2026-10-16 10:33:43.636393


Quick check of the guard-rail: The next cell tries a write and should be stopped by `q()` before anything reaches the server:


In [20]:
try:
    q("DELETE FROM week06.london_2014")
except PermissionError as e:
    print("Blocked as expected ->", e)


Blocked as expected -> 'DELETE' is not allowed: this notebook is read-only. Shape your data inside the SELECT, or in pandas.


And the server's own guard-rail: Even if you bypass `q()` with a raw connection, the database refuses:


In [21]:
conn = get_connection()
try:
    cur = conn.cursor()
    cur.execute("UPDATE week06.london_2014 SET [Mean TemperatureC] = 0 WHERE 1 = 0")
    print("Unexpected: the server accepted the statement. Tell the admin the account isn't read-only.")
except Exception as e:
    print("Server refused, as expected ->", str(e)[:160])
finally:
    conn.rollback()
    conn.close()


Server refused, as expected -> (229, b"The UPDATE permission was denied on the object 'london_2014', database 'assessmentDB', schema 'week06'.DB-Lib error message 20018, severity 14:\nGeneral


---
## 1. Exploring an unfamiliar database
On day one of any data job, someone hands you a database you've never seen. Before you write any analysis, answer three questions: **what tables exist**, **what columns do they have**, and **what does the data actually look like**?

### 1.1 `INFORMATION_SCHEMA`: the database's own catalogue
Every SQL Server database describes itself through read-only system views. `TABLES` lists the tables and `COLUMNS` lists every column with its data type.


In [22]:
tables = q("""
SELECT TABLE_SCHEMA AS schema_name, TABLE_NAME AS table_name, TABLE_TYPE
FROM INFORMATION_SCHEMA.TABLES
ORDER BY TABLE_SCHEMA, TABLE_NAME
""")
print(len(tables), "tables")
tables.head(15)


67 tables


,schema_name,table_name,TABLE_TYPE
0,sys,database_firewall_rules,VIEW
1,week06,beijing_pek_2014,BASE TABLE
2,week06,brasilia_bsb_2014,BASE TABLE
3,week06,capetown_cpt_2014,BASE TABLE
4,week06,delhi_del_2014,BASE TABLE
5,week06,london_2014,BASE TABLE
6,week06,moscow_svo_2014,BASE TABLE
7,week06,who_pop_tb_all,BASE TABLE
8,week06,who_pop_tb_some,BASE TABLE
9,week07,london_2014,BASE TABLE


In [23]:
# Tables per schema (each schema is one week of the programme)
q("""
SELECT TABLE_SCHEMA AS schema_name, COUNT(*) AS n_tables
FROM INFORMATION_SCHEMA.TABLES
GROUP BY TABLE_SCHEMA
ORDER BY TABLE_SCHEMA
""")


,schema_name,n_tables
0,sys,1
1,week06,8
2,week07,1
3,week08,3
4,week09,3
5,week14,10
6,week15,8
7,week16,9
8,week17,8
9,week18,7


In [24]:
# Every column of the London weather table, with its type, in table order
q("""
SELECT ORDINAL_POSITION AS pos, COLUMN_NAME, DATA_TYPE, IS_NULLABLE
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'week06' AND TABLE_NAME = 'london_2014'
ORDER BY ORDINAL_POSITION
""")


,pos,COLUMN_NAME,DATA_TYPE,IS_NULLABLE
0,1,GMT,varchar,YES
1,2,Max TemperatureC,bigint,YES
2,3,Mean TemperatureC,bigint,YES
3,4,Min TemperatureC,bigint,YES
4,5,Dew PointC,bigint,YES
5,6,MeanDew PointC,bigint,YES
6,7,Min DewpointC,bigint,YES
7,8,Max Humidity,bigint,YES
8,9,Mean Humidity,bigint,YES
9,10,Min Humidity,bigint,YES


### 1.2 ⚠️ Hidden spaces in column names (a real problem in this database)
These tables were loaded straight from CSV files, and some CSV headers had a space after the comma. The weather tables therefore contain columns whose names start with a space: `" Events"`, `" Mean Humidity"`, `" CloudCover"`. You can't see the space when you print the table, but SQL needs it, so you must write `[ Events]`, not `[Events]`.

Find them like this. Wrapping each name in brackets makes the space visible:


In [25]:
q("""
SELECT DISTINCT '[' + COLUMN_NAME + ']' AS exact_name_to_use
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'week06' AND COLUMN_NAME LIKE ' %' -- starts with a space
ORDER BY exact_name_to_use
""")


,exact_name_to_use
0,[ CloudCover]
1,[ Events]
2,[ Max Gust SpeedKm/h]
3,[ Max Sea Level PressurehPa]
4,[ Max VisibilityKm]
5,[ Max Wind SpeedKm/h]
6,[ Mean Humidity]
7,[ Mean Sea Level PressurehPa]
8,[ Mean VisibilityKm]
9,[ Mean Wind SpeedKm/h]


### 1.3 Find a column anywhere in the database
Which tables have an `Age` column? Or anything about `income`?


In [26]:
q("""
SELECT TABLE_SCHEMA + '.' + TABLE_NAME AS table_name, COLUMN_NAME, DATA_TYPE
FROM INFORMATION_SCHEMA.COLUMNS
WHERE COLUMN_NAME LIKE '%Income%' OR COLUMN_NAME = 'Age'
ORDER BY table_name
""")


,table_name,COLUMN_NAME,DATA_TYPE
0,week14.assignment1_ecommerce_data,Age,float
1,week14.assignment1_ecommerce_data,Annual_Income,float
2,week14.data,Age,float
3,week14.housing_price_data,Age,bigint
4,week14.task1_data_with_missing,Age,float
5,week14.task1_data_with_missing,Income,float
6,week14.task2_categorical_data,Age,bigint
7,week14.task3_scaling_data,Age,bigint
8,week16.assignment1_house_prices,Age,bigint
9,week16.assignment2_marketing_campaign,Age,bigint


### 1.4 Profile a table before you trust it
A quick profile tells you the row count, the number of distinct values, and how many values are missing. Make this a habit.


In [27]:
q("""
SELECT 
    COUNT(*) AS n_rows,
    COUNT(DISTINCT Gender) AS n_gender_values,
    SUM(CASE WHEN Gender IS NULL THEN 1 ELSE 0 END) AS missing_gender,
    SUM(CASE WHEN Applicant_Income IS NULL THEN 1 ELSE 0 END) AS missing_income,
    SUM(CASE WHEN Loan_Amount IS NULL THEN 1 ELSE 0 END) AS missing_loan_amount,
    MIN(Applicant_Income) AS min_income,
    MAX(Applicant_Income) AS max_income
FROM week17.loan_approval_prediction
""")


,n_rows,n_gender_values,missing_gender,missing_income,missing_loan_amount,min_income,max_income
0,500,2,0,0,0,17000,80000


### 1.5 Peek, don't dump
`SELECT TOP 5 *` is the SQL version of `df.head()`. When you're exploring, never pull a whole table just to look at it. On a real warehouse, a table can have billions of rows.


In [28]:
q("SELECT TOP 5 * FROM week09.comtrade_milk_uk_monthly_14")


,Classification,Year,Period,Period Desc.,Aggregate Level,Is Leaf Code,Trade Flow Code,Trade Flow,Reporter Code,Reporter,Reporter ISO,Partner Code,Partner,Partner ISO,2nd Partner Code,2nd Partner,2nd Partner ISO,Customs Proc. Code,Customs,Mode of Transport Code,Mode of Transport,Commodity Code,Commodity,Qty Unit Code,Qty Unit,Qty,Alt Qty Unit Code,Alt Qty Unit,Alt Qty,Netweight (kg),Gross weight (kg),Trade Value (US$),CIF Trade Value (US$),FOB Trade Value (US$),Flag
0,HS,2014,201401,January 2014,4,0,1,Imports,826,United Kingdom,None,0,World,None,None,None,None,None,None,None,None,0401,"Milk and cream, neither concentrated nor sweet...",0,None,None,None,None,None,22404316,None,21950747,None,None,0
1,HS,2014,201401,January 2014,4,0,2,Exports,826,United Kingdom,None,0,World,None,None,None,None,None,None,None,None,0401,"Milk and cream, neither concentrated nor sweet...",0,None,None,None,None,None,60497363,None,46923551,None,None,0
2,HS,2014,201401,January 2014,4,0,2,Exports,826,United Kingdom,None,4,Afghanistan,None,None,None,None,None,None,None,None,0401,"Milk and cream, neither concentrated nor sweet...",0,None,None,None,None,None,2520,None,3410,None,None,0
3,HS,2014,201401,January 2014,4,0,2,Exports,826,United Kingdom,None,40,Austria,None,None,None,None,None,None,None,None,0401,"Milk and cream, neither concentrated nor sweet...",0,None,None,None,None,None,21,None,316,None,None,0
4,HS,2014,201401,January 2014,4,0,1,Imports,826,United Kingdom,None,56,Belgium,None,None,None,None,None,None,None,None,0401,"Milk and cream, neither concentrated nor sweet...",0,None,None,None,None,None,1978279,None,4472349,None,None,0


### Exercise 1.1
List every table in the `week16` and `week17` schemas together with how many columns each one has. Sort by column count, highest first.


In [29]:
# Solution 1.1
q("""
SELECT TABLE_SCHEMA + '.' + TABLE_NAME AS table_name, COUNT(*) AS n_columns
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA IN ('week16', 'week17')
GROUP BY TABLE_SCHEMA, TABLE_NAME
ORDER BY n_columns DESC
""")


,table_name,n_columns
0,week16.customer_churn_prediction,19
1,week17.loan_approval_prediction,16
2,week16.assignment3_credit_risk,10
3,week17.assignment2_medical_diagnosis,10
4,week16.assignment2_marketing_campaign,9
5,week16.task3_classification_data,9
6,week16.assignment1_house_prices,8
7,week17.assignment1_product_recommendation,8
8,week17.assignment3_classifier_comparison,7
9,week17.task3_naive_bayes_data,7


### Exercise 1.2
Profile `week16.customer_churn_prediction`: total rows, number of distinct `Contract_Type` values, number of churned customers (`Churn = 1`), and the min / max of `Monthly_Charges`.


In [30]:
# Solution 1.2
q("""
SELECT 
    COUNT(*) AS n_rows,
    COUNT(DISTINCT Contract_Type) AS n_contract_types,
    SUM(Churn) AS n_churned,
    MIN(Monthly_Charges) AS min_monthly,
    MAX(Monthly_Charges) AS max_monthly
FROM week16.customer_churn_prediction
""")


,n_rows,n_contract_types,n_churned,min_monthly,max_monthly
0,500,3,228,20.03,119.20


---
## 2. Refresher: the core SELECT, done properly

### 2.1 Anatomy and style
```sql
SELECT TOP 10                                     -- how many rows (T-SQL)
    country,                                      -- one column per line
    [NY.GDP.MKTP.CD] / 1e9 AS gdp_billion_usd     -- computed column with an alias
FROM week08.wb_gdp_2013 AS g                      -- table alias
WHERE [NY.GDP.MKTP.CD] IS NOT NULL                -- filter rows
ORDER BY gdp_billion_usd DESC;                    -- sort (aliases allowed here)
```
- SQL keywords in **UPPER CASE** and one clause per line.
- Name every computed column with `AS snake_case_name`.
- Bracket-quote any name with spaces or symbols: `[Trade Value (US$)]`.
- List the columns you need. Use `SELECT *` only when exploring.


In [31]:
q("""
SELECT TOP 10 
    country, 
    [NY.GDP.MKTP.CD] / 1e9 AS gdp_billion_usd
FROM week08.wb_gdp_2013
WHERE [NY.GDP.MKTP.CD] IS NOT NULL
ORDER BY gdp_billion_usd DESC
""")


,country,gdp_billion_usd
0,World,"76,236.80"
1,High income,"52,256.19"
2,OECD members,"48,009.76"
3,High income: OECD,"45,927.31"
4,Low & middle income,"24,011.49"
5,Middle income,"23,637.12"
6,Europe & Central Asia (all income levels),"22,949.90"
7,East Asia & Pacific (all income levels),"20,846.73"
8,North America,"18,612.59"
9,Upper middle income,"18,155.01"


### 2.2 DISTINCT: unique values of a column (or combination of columns)


In [32]:
q("""
SELECT DISTINCT [Trade Flow], [Commodity Code], Commodity
FROM week09.comtrade_milk_uk_monthly_14
ORDER BY [Trade Flow], [Commodity Code]
""")


,Trade Flow,Commodity Code,Commodity
0,Exports,0401,"Milk and cream, neither concentrated nor sweet..."
1,Exports,0402,"Milk and cream, concentrated or sweetened"
2,Imports,0401,"Milk and cream, neither concentrated nor sweet..."
3,Imports,0402,"Milk and cream, concentrated or sweetened"


### 2.3 TOP ... WITH TIES and TOP ... PERCENT
`TOP 5` cuts at exactly 5 rows, even when the 5th and 6th rows tie. `WITH TIES` keeps every row that ties with the last one. It needs an `ORDER BY`.


In [33]:
q("""
SELECT TOP 5 WITH TIES GMT, [Max TemperatureC]
FROM week06.london_2014
ORDER BY [Max TemperatureC] DESC
""")


,GMT,Max TemperatureC
0,2014-7-18,30
1,2014-7-23,29
2,2014-7-24,29
3,2014-7-26,29
4,2014-7-17,29


### 2.4 Paging with OFFSET ... FETCH
This is the standard SQL way to say *"skip 10 rows, then give me 5"*. Dashboards and APIs page through results like this.


In [34]:
q("""
SELECT country, [SP.POP.TOTL] AS population
FROM week08.wb_pop_2013
WHERE [SP.POP.TOTL] IS NOT NULL
ORDER BY population DESC
OFFSET 10 ROWS FETCH NEXT 5 ROWS ONLY  -- ranks 11 to 15
""")


,country,population
0,India,"1,279,498,874.00"
1,OECD members,"1,263,178,643.00"
2,High income: OECD,"1,064,428,332.00"
3,Sub-Saharan Africa (all income levels),"948,287,652.00"
4,Sub-Saharan Africa (developing only),"947,400,670.00"


### Exercise 2.1
Show the 10 least populous entries in `week08.wb_pop_2013` (ignore NULLs), with population shown in thousands and rounded to 1 decimal place, in a column called `pop_thousands`.


In [35]:
# Solution 2.1
q("""
SELECT TOP 10 country, ROUND([SP.POP.TOTL] / 1000.0, 1) AS pop_thousands
FROM week08.wb_pop_2013
WHERE [SP.POP.TOTL] IS NOT NULL
ORDER BY [SP.POP.TOTL] ASC
""")


,country,pop_thousands
0,Tuvalu,9.90
1,Palau,20.90
2,St. Martin (French part),31.30
3,San Marino,31.40
4,Turks and Caicos Islands,33.10
5,Sint Maarten (Dutch part),36.60
6,Liechtenstein,37.00
7,Monaco,37.50
8,Faeroe Islands,48.30
9,Marshall Islands,52.80


---
## 3. Filtering like a pro

### 3.1 IN, BETWEEN, NOT


In [36]:
q("""
SELECT Application_ID, Property_Area, Education, Applicant_Income, Loan_Amount, Loan_Approved
FROM week17.loan_approval_prediction
WHERE Property_Area IN ('Urban', 'Semiurban')
  AND Applicant_Income BETWEEN 5000 AND 10000 -- inclusive on both ends
  AND NOT Education = 'Graduate'
ORDER BY Applicant_Income
""").head(10)


,Application_ID,Property_Area,Education,Applicant_Income,Loan_Amount,Loan_Approved


### 3.2 LIKE pattern matching
| Pattern | Matches |
|---|---|
| `'Rain%'` | starts with Rain |
| `'%Rain%'` | contains Rain anywhere |
| `'%Rain'` | ends with Rain |
| `'_og%'` | any one character, then og (e.g. Fog) |
| `'[A-C]%'` | starts with A, B or C (T-SQL character class) |


In [37]:
SELECT_EVENTS = q("""
SELECT [ Events] AS events, COUNT(*) AS n_days
FROM week06.london_2014
WHERE [ Events] LIKE '%Rain%'
GROUP BY [ Events]
ORDER BY n_days DESC
""")
SELECT_EVENTS


,events,n_days
0,Rain,202
1,Fog-Rain,17
2,Rain-Thunderstorm,15
3,Fog-Rain-Thunderstorm,2
4,Fog-Rain-Snow-Thunderstorm,1
5,Rain-Snow,1


### 3.3 NULL means "unknown", and it breaks normal logic
`NULL` is not zero and not an empty string. It means we don't know. Any comparison with `NULL` returns unknown, and `WHERE` keeps only rows that are true. So:
- `WHERE Events = NULL` returns nothing. Write `WHERE Events IS NULL`.
- `WHERE Events <> 'Rain'` silently drops the NULL rows as well.

Let's prove it with the London data (365 days in total):


In [38]:
q("""
SELECT 
    (SELECT COUNT(*) FROM week06.london_2014) AS all_days,
    (SELECT COUNT(*) FROM week06.london_2014 WHERE [ Events] = 'Rain') AS rain_only,
    (SELECT COUNT(*) FROM week06.london_2014 WHERE [ Events] <> 'Rain') AS not_rain_naive,
    (SELECT COUNT(*) FROM week06.london_2014 WHERE [ Events] IS NULL) AS no_event_recorded,
    (SELECT COUNT(*) FROM week06.london_2014 WHERE [ Events] <> 'Rain' OR [ Events] IS NULL) AS not_rain_correct
""")


,all_days,rain_only,not_rain_naive,no_event_recorded,not_rain_correct
0,365,202,49,114,163


### 3.4 Handling NULLs: COALESCE, ISNULL, NULLIF
- `COALESCE(a, b, c)` returns the first non-NULL value (standard SQL).
- `ISNULL(a, b)` is the T-SQL two-argument version.
- `NULLIF(a, b)` returns `NULL` when `a = b`. Its main use is preventing **divide-by-zero**.


In [39]:
q("""
SELECT TOP 10 
    GMT, 
    COALESCE([ Events], 'No event') AS events_clean,
    Precipitationmm,
    [Max TemperatureC] - [Min TemperatureC] AS temp_range,
    -- NULLIF prevents divide-by-zero if range is 0
    Precipitationmm / NULLIF([Max TemperatureC] - [Min TemperatureC], 0) AS rain_per_degree_range
FROM week06.london_2014
ORDER BY Precipitationmm DESC
""")


,GMT,events_clean,Precipitationmm,temp_range,rain_per_degree_range
0,2014-8-8,Rain,21.08,7,3.01
1,2014-8-11,Rain-Thunderstorm,19.05,9,2.12
2,2014-7-28,Rain-Thunderstorm,17.02,9,1.89
3,2014-8-25,Rain,17.02,4,4.25
4,2014-1-29,Rain,13.97,2,6.99
5,2014-6-28,Rain,13.97,6,2.33
6,2014-11-23,Rain,13.97,5,2.79
7,2014-11-2,Rain,11.94,6,1.99
8,2014-11-4,Fog-Rain-Snow-Thunderstorm,11.94,10,1.19
9,2014-1-1,Rain,9.91,5,1.98


### Exercise 3.1
From `week17.loan_approval_prediction`, find self-employed applicants whose `Previous_Loan_Status` is `Defaulted` but who were still approved. Show ID, income, loan amount and credit history.


In [40]:
# Solution 3.1
q("""
SELECT Application_ID, Applicant_Income, Loan_Amount, Credit_History
FROM week17.loan_approval_prediction
WHERE Self_Employed = 'Yes'
  AND Previous_Loan_Status = 'Defaulted'
  AND Loan_Approved = 1
""")


,Application_ID,Applicant_Income,Loan_Amount,Credit_History


### Exercise 3.2
How many loan applications have at least one of `Gender`, `Married` or `Self_Employed` missing? (Hint: `OR` + `IS NULL`.)


In [41]:
# Solution 3.2
q("""
SELECT COUNT(*) AS rows_with_missing
FROM week17.loan_approval_prediction
WHERE Gender IS NULL OR Married IS NULL OR Self_Employed IS NULL
""")


,rows_with_missing
0,0


---
## 4. Data types, conversion & built-in functions

### 4.1 Know your types
- `CAST(x AS date)` / `CONVERT(date, x)`: errors and stops the whole query on bad input.
- `TRY_CAST(x AS date)` / `TRY_CONVERT(date, x)`: returns `NULL` for bad rows instead of crashing.


In [42]:
q("""
SELECT TOP 5 
    GMT AS raw_value,
    TRY_CONVERT(date, GMT) AS as_date,
    DATENAME(weekday, TRY_CONVERT(date, GMT)) AS day_name,
    MONTH(TRY_CONVERT(date, GMT)) AS month_no,
    DATENAME(month, TRY_CONVERT(date, GMT)) AS month_name,
    DATEPART(quarter, TRY_CONVERT(date, GMT)) AS qtr,
    EOMONTH(TRY_CONVERT(date, GMT)) AS month_end
FROM week06.london_2014
""")


,raw_value,as_date,day_name,month_no,month_name,qtr,month_end
0,2014-1-1,2014-01-01,Wednesday,1,January,1,2014-01-31
1,2014-1-2,2014-01-02,Thursday,1,January,1,2014-01-31
2,2014-1-3,2014-01-03,Friday,1,January,1,2014-01-31
3,2014-1-4,2014-01-04,Saturday,1,January,1,2014-01-31
4,2014-1-5,2014-01-05,Sunday,1,January,1,2014-01-31


### 4.2 Date arithmetic
`DATEADD(day, 7, d)`, `DATEDIFF(day, start, end)`, `DATEFROMPARTS(2014, 6, 4)`.


In [43]:
q("""
SELECT 
    TRY_CONVERT(date, GMT) AS day,
    [Mean TemperatureC],
    DATEDIFF(day, DATEFROMPARTS(2014, 1, 1), TRY_CONVERT(date, GMT)) + 1 AS day_of_year
FROM week06.london_2014
WHERE TRY_CONVERT(date, GMT) BETWEEN DATEFROMPARTS(2014, 12, 8) AND DATEFROMPARTS(2014, 12, 12)
ORDER BY day
""")


,day,Mean TemperatureC,day_of_year
0,2014-12-08,4,342
1,2014-12-09,6,343
2,2014-12-10,7,344
3,2014-12-11,8,345
4,2014-12-12,6,346


### 4.3 String functions
`LEN`, `UPPER`/`LOWER`, `LEFT`/`RIGHT`/`SUBSTRING`, `CHARINDEX`, `REPLACE`, `TRIM`, `CONCAT`.


In [44]:
q("""
SELECT DISTINCT TOP 5 
    Period,
    DATEFROMPARTS(Period / 100, Period % 100, 1) AS period_start,
    CONCAT(DATENAME(month, DATEFROMPARTS(Period / 100, Period % 100, 1)), ' ', Period / 100) AS period_label
FROM week09.comtrade_milk_uk_monthly_14
ORDER BY Period
""")


,Period,period_start,period_label
0,201401,2014-01-01,January 2014
1,201402,2014-02-01,February 2014
2,201403,2014-03-01,March 2014
3,201404,2014-04-01,April 2014
4,201405,2014-05-01,May 2014


In [45]:
q("""
SELECT DISTINCT 
    Commodity,
    LEN(Commodity) AS n_chars,
    LEFT(Commodity, CHARINDEX(',', Commodity + ',') - 1) AS short_name,
    UPPER(REPLACE(Commodity, 'Milk and cream', 'M&C')) AS abbreviated
FROM week09.comtrade_milk_uk_monthly_14
""")


,Commodity,n_chars,short_name,abbreviated
0,"Milk and cream, concentrated or sweetened",41,Milk and cream,"M&C, CONCENTRATED OR SWEETENED"
1,"Milk and cream, neither concentrated nor sweet...",50,Milk and cream,"M&C, NEITHER CONCENTRATED NOR SWEETENED"


### 4.4 ⚠️ The integer-division trap
In SQL Server, `integer ÷ integer = integer`, so fractional parts are discarded. `Loan_Approved` is stored as an integer (0/1), so `AVG(Loan_Approved)` returns 0! Multiply by `1.0` or `CAST(... AS float)` first.


In [46]:
q("""
SELECT 
    AVG(Loan_Approved) AS wrong_rate,       -- integer average, truncated to 0
    AVG(CAST(Loan_Approved AS float)) AS right_rate_cast,
    AVG(1.0 * Loan_Approved) AS right_rate_trick,
    7 / 2 AS int_division,                  -- 3
    7 / 2.0 AS dec_division                 -- 3.5
FROM week17.loan_approval_prediction
""")


,wrong_rate,right_rate_cast,right_rate_trick,int_division,dec_division
0,0,0.67,0.67,3,3.50


### 4.5 Maths: ROUND, CEILING, FLOOR, ABS, POWER, SQRT, LOG10


In [47]:
q("""
SELECT TOP 5 
    country, 
    [NY.GDP.MKTP.CD] AS gdp,
    ROUND([NY.GDP.MKTP.CD] / 1e9, 2) AS gdp_bn,
    FLOOR(LOG10([NY.GDP.MKTP.CD])) AS order_of_magnitude
FROM week08.wb_gdp_2013
WHERE [NY.GDP.MKTP.CD] > 0
ORDER BY gdp DESC
""")


,country,gdp,gdp_bn,order_of_magnitude
0,World,"76,236,796,175,538.20","76,236.80",13.00
1,High income,"52,256,187,329,042.10","52,256.19",13.00
2,OECD members,"48,009,763,073,164.40","48,009.76",13.00
3,High income: OECD,"45,927,311,632,922.50","45,927.31",13.00
4,Low & middle income,"24,011,494,861,267.10","24,011.49",13.00


### Exercise 4.1
For London, count how many days fell on each weekday (Monday...Sunday) where `Precipitationmm > 0`. Sort by the number of wet days, most first.


In [48]:
# Solution 4.1
q("""
SELECT 
    DATENAME(weekday, TRY_CONVERT(date, GMT)) AS weekday,
    COUNT(*) AS wet_days
FROM week06.london_2014
WHERE Precipitationmm > 0
GROUP BY DATENAME(weekday, TRY_CONVERT(date, GMT))
ORDER BY wet_days DESC
""")


,weekday,wet_days
0,Monday,30
1,Friday,29
2,Tuesday,28
3,Wednesday,24
4,Saturday,23
5,Thursday,19
6,Sunday,17


### Exercise 4.2
Compute the churn rate (as a percentage with 1 decimal place) in `week16.customer_churn_prediction`. Avoid the integer-division trap!


In [49]:
# Solution 4.2
q("""
SELECT ROUND(100.0 * SUM(Churn) / COUNT(*), 1) AS churn_rate_pct
FROM week16.customer_churn_prediction
""")


,churn_rate_pct
0,45.60


---
## 5. CASE expressions: if/else inside SQL
`CASE` is the SQL equivalent of `np.where` / `pd.cut`. It is evaluated top to bottom and the first true branch wins.

### 5.1 Bucketing a numeric column


In [50]:
q("""
SELECT 
    CASE 
        WHEN Applicant_Income < 3000 THEN '1. < 3k'
        WHEN Applicant_Income < 6000 THEN '2. 3k-6k'
        WHEN Applicant_Income < 10000 THEN '3. 6k-10k'
        WHEN Applicant_Income IS NULL THEN '9. unknown'
        ELSE '4. 10k+'
    END AS income_band,
    COUNT(*) AS applicants,
    ROUND(100.0 * AVG(1.0 * Loan_Approved), 1) AS approval_rate_pct
FROM week17.loan_approval_prediction
GROUP BY 
    CASE 
        WHEN Applicant_Income < 3000 THEN '1. < 3k'
        WHEN Applicant_Income < 6000 THEN '2. 3k-6k'
        WHEN Applicant_Income < 10000 THEN '3. 6k-10k'
        WHEN Applicant_Income IS NULL THEN '9. unknown'
        ELSE '4. 10k+'
    END
ORDER BY income_band
""")


,income_band,applicants,approval_rate_pct
0,4. 10k+,500,66.80


### 5.2 Conditional aggregation: the most useful pattern in analytics
Put a `CASE` inside an aggregate to compute several filtered metrics in a single pass.


In [51]:
q("""
SELECT 
    Contract_Type,
    COUNT(*) AS customers,
    SUM(CASE WHEN Churn = 1 THEN 1 ELSE 0 END) AS churned,
    ROUND(100.0 * AVG(CASE WHEN Churn = 1 THEN 1.0 ELSE 0 END), 1) AS churn_rate_pct,
    ROUND(AVG(CASE WHEN Churn = 1 THEN Monthly_Charges END), 2) AS avg_charge_churned,
    ROUND(AVG(CASE WHEN Churn = 0 THEN Monthly_Charges END), 2) AS avg_charge_stayed
FROM week16.customer_churn_prediction
GROUP BY Contract_Type
ORDER BY churn_rate_pct DESC
""")


,Contract_Type,customers,churned,churn_rate_pct,avg_charge_churned,avg_charge_stayed
0,Month-to-Month,158,91,57.60,65.03,62.86
1,One Year,165,89,53.90,61.22,63.01
2,Two Year,177,48,27.10,70.76,62.79


### 5.3 IIF: a shortcut for a two-way CASE (T-SQL only)


In [52]:
q("""
SELECT 
    IIF([Mean TemperatureC] >= 15, 'warm', 'cool') AS day_type,
    COUNT(*) AS n_days,
    ROUND(AVG(Precipitationmm), 2) AS avg_rain_mm
FROM week06.london_2014
GROUP BY IIF([Mean TemperatureC] >= 15, 'warm', 'cool')
""")


,day_type,n_days,avg_rain_mm
0,cool,240,1.51
1,warm,125,1.12


### Exercise 5.1
Classify churn customers by `Tenure` into `'New (<12m)'`, `'Established (12-36m)'` and `'Loyal (36m+)'`. For each group show customer count and churn rate %.


In [53]:
# Solution 5.1
q("""
SELECT 
    CASE 
        WHEN Tenure < 12 THEN 'New (<12m)'
        WHEN Tenure < 36 THEN 'Established (12-36m)'
        ELSE 'Loyal (36m+)'
    END AS tenure_group,
    COUNT(*) AS customers,
    ROUND(100.0 * AVG(1.0 * Churn), 1) AS churn_rate_pct
FROM week16.customer_churn_prediction
GROUP BY 
    CASE 
        WHEN Tenure < 12 THEN 'New (<12m)'
        WHEN Tenure < 36 THEN 'Established (12-36m)'
        ELSE 'Loyal (36m+)'
    END
ORDER BY churn_rate_pct DESC
""")


,tenure_group,customers,churn_rate_pct
0,New (<12m),73,57.50
1,Established (12-36m),175,45.70
2,Loyal (36m+),252,42.10


### Exercise 5.2
Using conditional aggregation, produce one row per `Property_Area` with columns: `applications`, `approved`, `rejected`, `approved_with_bad_credit` (`approved AND Credit_History = 0`).


In [54]:
# Solution 5.2
q("""
SELECT 
    Property_Area,
    COUNT(*) AS applications,
    SUM(CASE WHEN Loan_Approved = 1 THEN 1 ELSE 0 END) AS approved,
    SUM(CASE WHEN Loan_Approved = 0 THEN 1 ELSE 0 END) AS rejected,
    SUM(CASE WHEN Loan_Approved = 1 AND Credit_History = 0 THEN 1 ELSE 0 END) AS approved_with_bad_credit
FROM week17.loan_approval_prediction
GROUP BY Property_Area
""")


,Property_Area,applications,approved,rejected,approved_with_bad_credit
0,Rural,166,0,166,0
1,Semiurban,167,167,0,0
2,Urban,167,167,0,0


---
## 6. Aggregation in depth

### 6.1 The three COUNTs
- `COUNT(*)`: all rows
- `COUNT(col)`: rows where col is not NULL
- `COUNT(DISTINCT col)`: unique non-NULL values


In [55]:
q("""
SELECT 
    COUNT(*) AS all_rows,
    COUNT([ Events]) AS rows_with_an_event,
    COUNT(DISTINCT [ Events]) AS distinct_event_types
FROM week06.london_2014
""")


,all_rows,rows_with_an_event,distinct_event_types
0,365,251,7


### 6.2 GROUP BY several columns, then HAVING
`WHERE` filters rows before grouping. `HAVING` filters groups after aggregation.


In [56]:
q("""
SELECT 
    Partner,
    COUNT(DISTINCT Period) AS active_months,
    SUM([Trade Value (US$)]) AS total_value_usd
FROM week09.comtrade_milk_uk_monthly_14
WHERE [Trade Flow] = 'Exports'
  AND Partner <> 'World'
GROUP BY Partner
HAVING COUNT(DISTINCT Period) >= 10
ORDER BY total_value_usd DESC
""").head(10)


,Partner,active_months,total_value_usd
0,Ireland,12,421228441
1,Algeria,12,93753841
2,Netherlands,12,68649082
3,Germany,12,36096633
4,France,12,34622982
5,Belgium,12,29343007
6,China,12,27317861
7,"China, Hong Kong SAR",12,19847947
8,Côte d'Ivoire,11,17732776
9,Angola,11,13305815


### 6.3 Statistical aggregates: STDEV and VAR


In [57]:
q("""
SELECT 
    Education,
    COUNT(*) AS n,
    ROUND(AVG(Applicant_Income), 0) AS mean_income,
    ROUND(STDEV(Applicant_Income), 0) AS sd_income,
    MIN(Applicant_Income) AS min_income,
    MAX(Applicant_Income) AS max_income
FROM week17.loan_approval_prediction
GROUP BY Education
""")


,Education,n,mean_income,sd_income,min_income,max_income
0,Graduate,334,52796,"14,010.00",32000,80000
1,Not Graduate,166,21487,"3,139.00",17000,28000


### 6.4 Subtotals and grand totals with ROLLUP
`GROUP BY ROLLUP(a, b)` returns normal groups plus subtotals and grand totals.


In [58]:
q("""
SELECT 
    CASE WHEN GROUPING([Trade Flow]) = 1 THEN '** ALL FLOWS **' ELSE [Trade Flow] END AS trade_flow,
    CASE WHEN GROUPING([Commodity Code]) = 1 THEN '(subtotal)' ELSE [Commodity Code] END AS commodity_code,
    SUM([Trade Value (US$)]) AS total_value_usd
FROM week09.comtrade_milk_uk_monthly_14
WHERE Partner <> 'World'
GROUP BY ROLLUP([Trade Flow], [Commodity Code])
ORDER BY GROUPING([Trade Flow]), [Trade Flow], GROUPING([Commodity Code]), [Commodity Code]
""")


,trade_flow,commodity_code,total_value_usd
0,Exports,0401,433790652
1,Exports,0402,464861283
2,Exports,(subtotal),898651935
3,Imports,0401,222107771
4,Imports,0402,341777171
5,Imports,(subtotal),563884942
6,** ALL FLOWS **,(subtotal),1462536877


### 6.5 STRING_AGG: collapse many rows into one string


In [59]:
q("""
SELECT 
    Region,
    COUNT(*) AS customers,
    STRING_AGG(Preferred_Category, ', ') WITHIN GROUP (ORDER BY Preferred_Category) AS categories
FROM (SELECT DISTINCT Region, Preferred_Category FROM week19.customer_analytics_data) AS d
GROUP BY Region
""")


,Region,customers,categories
0,East,6,"Beauty, Books, Electronics, Fashion, Home & Ki..."
1,North,6,"Beauty, Books, Electronics, Fashion, Home & Ki..."
2,South,6,"Beauty, Books, Electronics, Fashion, Home & Ki..."
3,West,6,"Beauty, Books, Electronics, Fashion, Home & Ki..."


### Exercise 6.1
For `week19.customer_analytics_data`, show each `Region × Payment_Method` with the number of customers and average `Avg_Order_Value` (2 dp). Keep only combinations with more than 25 customers.


In [60]:
# Solution 6.1
q("""
SELECT 
    Region, 
    Payment_Method, 
    COUNT(*) AS customers,
    ROUND(AVG(Avg_Order_Value), 2) AS avg_order_value
FROM week19.customer_analytics_data
GROUP BY Region, Payment_Method
HAVING COUNT(*) > 25
ORDER BY Region, customers DESC
""")


,Region,Payment_Method,customers,avg_order_value
0,East,Debit Card,40,489.52
1,East,Credit Card,36,492.20
2,North,Credit Card,45,486.00
3,North,Debit Card,42,483.67
4,North,Digital Wallet,35,490.04
5,South,Credit Card,39,484.96
6,South,Digital Wallet,37,499.73
7,West,Debit Card,39,484.94
8,West,Credit Card,35,482.14
9,West,Digital Wallet,26,476.28


### Exercise 6.2
Use `ROLLUP` on `week16.customer_churn_prediction` to show churned customers by `Internet_Service` and `Contract_Type`, with subtotals per internet service and a grand total.


In [61]:
# Solution 6.2
q("""
SELECT 
    COALESCE(Internet_Service, '** TOTAL **') AS internet_service,
    COALESCE(Contract_Type, '(all)') AS contract_type,
    SUM(Churn) AS churned
FROM week16.customer_churn_prediction
GROUP BY ROLLUP(Internet_Service, Contract_Type)
""")


,internet_service,contract_type,churned
0,DSL,Month-to-Month,28
1,DSL,One Year,33
2,DSL,Two Year,13
3,DSL,(all),74
4,Fiber Optic,Month-to-Month,44
5,Fiber Optic,One Year,32
6,Fiber Optic,Two Year,25
7,Fiber Optic,(all),101
8,No,Month-to-Month,19
9,No,One Year,24


---
## 7. Joins

### 7.1 Which join do I need?
| Join | Keeps | Use it for |
|---|---|---|
| **INNER JOIN** | only rows that match on both sides | *"facts that exist in both datasets"* |
| **LEFT JOIN** | all rows from left, matches from right (NULL if none) | *"enrich my main table, never lose rows"* |
| **LEFT JOIN ... WHERE right.key IS NULL** | left rows with no match (anti-join) | data-quality checks, *"who's missing?"* |
| **FULL OUTER JOIN** | everything from both sides | reconciling two sources |
| **CROSS JOIN** | every combination | building grids or scaffolds |

### 7.2 Inner join of three tables (World Bank GDP, Population, Life Expectancy)


In [62]:
wb = q("""
SELECT 
    g.country,
    g.[NY.GDP.MKTP.CD] AS gdp_usd,
    p.[SP.POP.TOTL] AS population,
    g.[NY.GDP.MKTP.CD] / NULLIF(p.[SP.POP.TOTL], 0) AS gdp_per_capita,
    l.[SP.DYN.LE00.IN] AS life_expectancy
FROM week08.wb_gdp_2013 AS g
JOIN week08.wb_pop_2013 AS p ON p.country = g.country
JOIN week08.wb_le_2013  AS l ON l.country = g.country
WHERE g.[NY.GDP.MKTP.CD] IS NOT NULL
""")
print(wb.shape)
wb.sort_values("gdp_per_capita", ascending=False).head(10)


(225, 5)


,country,gdp_usd,population,gdp_per_capita,life_expectancy
121,Luxembourg,"61,794,506,555.51","543,360.00","113,726.64",81.80
152,Norway,"522,349,106,382.98","5,079,623.00","102,832.26",81.45
167,Qatar,"201,885,439,560.44","2,101,288.00","96,077.00",78.42
122,"Macao SAR, China","51,313,531,848.85","568,056.00","90,331.82",80.34
20,Bermuda,"5,573,710,000.00","65,001.00","85,748.07",80.57
198,Switzerland,"684,919,206,141.13","8,089,346.00","84,669.29",82.75
9,Australia,"1,563,950,959,269.52","23,125,868.00","67,627.77",82.20
197,Sweden,"578,742,001,487.57","9,600,379.00","60,283.25",81.70
50,Denmark,"335,877,548,363.83","5,614,932.00","59,818.63",80.30
178,Singapore,"302,245,904,259.57","5,399,200.00","55,979.76",82.35


### 7.3 Joining two different sources: WHO vs World Bank


In [63]:
q("""
SELECT 
    w.Country,
    w.[TB deaths],
    p.[SP.POP.TOTL] AS wb_population,
    ROUND(100000.0 * w.[TB deaths] / NULLIF(p.[SP.POP.TOTL], 0), 1) AS tb_deaths_per_100k
FROM week06.who_pop_tb_all AS w
JOIN week08.wb_pop_2013 AS p ON p.country = w.Country
ORDER BY tb_deaths_per_100k DESC
""").head(15)


,Country,TB deaths,wb_population,tb_deaths_per_100k
0,Djibouti,870.00,"864,554.00",100.60
1,Nigeria,"160,000.00","172,816,517.00",92.60
2,Swaziland,"1,100.00","1,250,641.00",88.00
3,Timor-Leste,990.00,"1,180,069.00",83.90
4,Somalia,"7,700.00","10,268,157.00",75.00
5,Guinea-Bissau,"1,200.00","1,757,138.00",68.30
6,Mozambique,"18,000.00","26,467,180.00",68.00
7,Cambodia,"10,000.00","15,078,564.00",66.30
8,Namibia,"1,300.00","2,346,592.00",55.40
9,Gabon,910.00,"1,650,351.00",55.10


### 7.4 The anti-join: find what didn't match


In [64]:
# WHO countries with no matching World Bank entry
q("""
SELECT w.Country AS who_country_with_no_wb_match
FROM week06.who_pop_tb_all AS w
LEFT JOIN week08.wb_pop_2013 AS p ON p.country = w.Country
WHERE p.country IS NULL
ORDER BY w.Country
""")


,who_country_with_no_wb_match
0,Bahamas
1,Bolivia (Plurinational State of)
2,Congo
3,Cook Islands
4,Côte d'Ivoire
5,Democratic People's Republic of Korea
6,Democratic Republic of the Congo
7,Egypt
8,Gambia
9,Iran (Islamic Republic of)


In [65]:
# World Bank entries not present in WHO (regional/income aggregates)
q("""
SELECT p.country AS wb_entry_not_in_who
FROM week08.wb_pop_2013 AS p
WHERE NOT EXISTS (
    SELECT 1 FROM week06.who_pop_tb_all AS w WHERE w.Country = p.country
)
ORDER BY p.country
""").head(15)


,wb_entry_not_in_who
0,American Samoa
1,Arab World
2,Aruba
3,"Bahamas, The"
4,Bermuda
5,Bolivia
6,Caribbean small states
7,Cayman Islands
8,Central Europe and the Baltics
9,Channel Islands


### 7.5 LEFT JOIN: keep every row of your main table


In [66]:
q("""
SELECT 
    w.Country,
    w.[Population (1000s)] AS who_pop_thousands,
    p.[SP.POP.TOTL] / 1000.0 AS wb_pop_thousands
FROM week06.who_pop_tb_all AS w
LEFT JOIN week08.wb_pop_2013 AS p ON p.country = w.Country
ORDER BY CASE WHEN p.country IS NULL THEN 0 ELSE 1 END, w.Country
""").head(10)


,Country,who_pop_thousands,wb_pop_thousands
0,Bahamas,377,NaN
1,Bolivia (Plurinational State of),10671,NaN
2,Congo,4448,NaN
3,Cook Islands,21,NaN
4,Côte d'Ivoire,20316,NaN
5,Democratic People's Republic of Korea,24895,NaN
6,Democratic Republic of the Congo,67514,NaN
7,Egypt,82056,NaN
8,Gambia,1849,NaN
9,Iran (Islamic Republic of),77447,NaN


### 7.6 FULL OUTER JOIN: reconcile two sources


In [67]:
q("""
SELECT 
    SUM(CASE WHEN w.Country IS NOT NULL AND p.country IS NOT NULL THEN 1 ELSE 0 END) AS in_both,
    SUM(CASE WHEN p.country IS NULL THEN 1 ELSE 0 END) AS only_in_who,
    SUM(CASE WHEN w.Country IS NULL THEN 1 ELSE 0 END) AS only_in_wb
FROM week06.who_pop_tb_all AS w
FULL OUTER JOIN week08.wb_pop_2013 AS p ON p.country = w.Country
""")


,in_both,only_in_who,only_in_wb
0,166,28,82


### 7.7 Self-join: compare each London day with the same day 7 days later


In [68]:
q("""
SELECT TOP 10 
    TRY_CONVERT(date, a.GMT) AS day,
    a.[Mean TemperatureC] AS temp,
    b.[Mean TemperatureC] AS temp_week_later,
    b.[Mean TemperatureC] - a.[Mean TemperatureC] AS change
FROM week06.london_2014 AS a
JOIN week06.london_2014 AS b 
  ON TRY_CONVERT(date, b.GMT) = DATEADD(day, 7, TRY_CONVERT(date, a.GMT))
ORDER BY ABS(b.[Mean TemperatureC] - a.[Mean TemperatureC]) DESC
""")


,day,temp,temp_week_later,change
0,2014-12-22,12,1,-11
1,2014-10-30,16,6,-10
2,2014-12-23,11,1,-10
3,2014-05-19,20,11,-9
4,2014-09-28,19,10,-9
5,2014-03-26,6,14,8
6,2014-10-27,16,8,-8
7,2014-11-29,10,2,-8
8,2014-03-16,13,6,-7
9,2014-03-23,6,13,7


### 7.8 ⚠️ Row explosion: joining on a key that isn't unique


In [69]:
q("""
SELECT 
    (SELECT COUNT(*) FROM week09.comtrade_milk_uk_monthly_14) AS rows_in_table,
    (SELECT COUNT(*) FROM week09.comtrade_milk_uk_monthly_14 a 
     JOIN week09.comtrade_milk_uk_monthly_14 b ON a.Partner = b.Partner) AS rows_after_bad_join
""")


,rows_in_table,rows_after_bad_join
0,1477,39065


### 7.9 CROSS JOIN: build a complete grid


In [70]:
q("""
WITH regions AS (
    SELECT DISTINCT Region FROM week19.customer_analytics_data
),
categories AS (
    SELECT DISTINCT Preferred_Category FROM week19.customer_analytics_data
)
SELECT 
    r.Region, 
    c.Preferred_Category, 
    COUNT(d.Customer_ID) AS customers
FROM regions r
CROSS JOIN categories c
LEFT JOIN week19.customer_analytics_data d 
  ON d.Region = r.Region AND d.Preferred_Category = c.Preferred_Category
GROUP BY r.Region, c.Preferred_Category
ORDER BY r.Region, customers DESC
""").head(15)


,Region,Preferred_Category,customers
0,East,Home & Kitchen,31
1,East,Fashion,27
2,East,Electronics,24
3,East,Books,19
4,East,Sports,10
5,East,Beauty,9
6,North,Electronics,40
7,North,Home & Kitchen,33
8,North,Sports,25
9,North,Fashion,23


### Exercise 7.1
Join GDP and life expectancy (`week08`) and list the 10 entries with the highest life expectancy whose GDP is below 50 billion USD. Show GDP in billions (1 dp).


In [71]:
# Solution 7.1
q("""
SELECT TOP 10 
    g.country, 
    ROUND(g.[NY.GDP.MKTP.CD] / 1e9, 1) AS gdp_bn,
    l.[SP.DYN.LE00.IN] AS life_expectancy
FROM week08.wb_gdp_2013 g
JOIN week08.wb_le_2013  l ON l.country = g.country
WHERE g.[NY.GDP.MKTP.CD] < 50e9 
  AND l.[SP.DYN.LE00.IN] IS NOT NULL
ORDER BY life_expectancy DESC
""")


,country,gdp_bn,life_expectancy
0,Iceland,15.40,83.12
1,Faeroe Islands,2.60,81.30
2,Malta,9.60,80.75
3,Bermuda,5.60,80.57
4,Slovenia,47.70,80.28
5,Lebanon,44.40,80.13
6,Cyprus,24.10,79.95
7,Costa Rica,49.20,79.23
8,Brunei Darussalam,16.10,78.55
9,Albania,12.80,77.54


### Exercise 7.2
Which WHO countries in `week06.who_pop_tb_all` have no life-expectancy value in `week08.wb_le_2013`, either because there's no matching row or because the value is NULL?


In [72]:
# Solution 7.2
q("""
SELECT 
    w.Country,
    CASE WHEN l.country IS NULL THEN 'no matching row' ELSE 'value is NULL' END AS reason
FROM week06.who_pop_tb_all w
LEFT JOIN week08.wb_le_2013 l ON l.country = w.Country
WHERE l.[SP.DYN.LE00.IN] IS NULL
ORDER BY reason, w.Country
""")


,Country,reason
0,Bahamas,no matching row
1,Bolivia (Plurinational State of),no matching row
2,Congo,no matching row
3,Cook Islands,no matching row
4,Côte d'Ivoire,no matching row
5,Democratic People's Republic of Korea,no matching row
6,Democratic Republic of the Congo,no matching row
7,Egypt,no matching row
8,Gambia,no matching row
9,Iran (Islamic Republic of),no matching row


---
## 8. Set operators: stacking and comparing result sets
- `UNION ALL`: all rows from both (fast, keeps duplicates). The SQL version of `pd.concat`.
- `UNION`: all rows, duplicates removed.
- `INTERSECT`: rows present in both.
- `EXCEPT`: rows in first but not second.

### 8.1 Compare column schemas with EXCEPT


In [73]:
q("""
SELECT 'only in london' AS side, COLUMN_NAME
FROM (
    SELECT COLUMN_NAME FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_SCHEMA = 'week06' AND TABLE_NAME = 'london_2014'
    EXCEPT
    SELECT COLUMN_NAME FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_SCHEMA = 'week06' AND TABLE_NAME = 'delhi_del_2014'
) a
UNION ALL
SELECT 'only in delhi', COLUMN_NAME
FROM (
    SELECT COLUMN_NAME FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_SCHEMA = 'week06' AND TABLE_NAME = 'delhi_del_2014'
    EXCEPT
    SELECT COLUMN_NAME FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_SCHEMA = 'week06' AND TABLE_NAME = 'london_2014'
) b
""")


,side,COLUMN_NAME
0,only in london,GMT
1,only in delhi,Date


### 8.2 UNION ALL: combine the six city tables
> **Note:** London's date column is named `GMT`, while the other five tables use `Date`.


In [74]:
cities = q("""
SELECT 'London' AS city, TRY_CONVERT(date, GMT) AS day, [Mean TemperatureC] AS mean_temp FROM week06.london_2014
UNION ALL
SELECT 'Beijing', TRY_CONVERT(date, [Date]), [Mean TemperatureC] FROM week06.beijing_pek_2014
UNION ALL
SELECT 'Brasilia', TRY_CONVERT(date, [Date]), [Mean TemperatureC] FROM week06.brasilia_bsb_2014
UNION ALL
SELECT 'Cape Town', TRY_CONVERT(date, [Date]), [Mean TemperatureC] FROM week06.capetown_cpt_2014
UNION ALL
SELECT 'Delhi', TRY_CONVERT(date, [Date]), [Mean TemperatureC] FROM week06.delhi_del_2014
UNION ALL
SELECT 'Moscow', TRY_CONVERT(date, [Date]), [Mean TemperatureC] FROM week06.moscow_svo_2014
""")
print(cities.shape) # expect (2190, 3) = 6 cities × 365 days
cities.groupby("city")["mean_temp"].agg(["mean", "min", "max"]).round(1)


(2190, 3)


,mean,min,max
city,,,
Beijing,13.40,-7,31
Brasilia,22.90,16,28
Cape Town,17.60,7,28
Delhi,13.70,-11,30
London,12.30,1,24
Moscow,6.00,-21,26


### 8.3 UNION vs UNION ALL: a counting example


In [75]:
q("""
SELECT 
    (SELECT COUNT(*) FROM (
        SELECT Partner FROM week09.comtrade_milk_uk_monthly_14
        UNION ALL
        SELECT Partner FROM week09.comtrade_milk_uk_jan_jul_15
    ) x) AS union_all_rows,
    (SELECT COUNT(*) FROM (
        SELECT Partner FROM week09.comtrade_milk_uk_monthly_14
        UNION
        SELECT Partner FROM week09.comtrade_milk_uk_jan_jul_15
    ) x) AS union_distinct_partners
""")


,union_all_rows,union_distinct_partners
0,2114,128


### 8.4 INTERSECT and EXCEPT: compare two periods (lost, new, kept export markets)


In [76]:
lost = q("""
SELECT Partner FROM week09.comtrade_milk_uk_monthly_14 WHERE [Trade Flow] = 'Exports'
EXCEPT
SELECT Partner FROM week09.comtrade_milk_uk_jan_jul_15 WHERE [Trade Flow] = 'Exports'
""")
new = q("""
SELECT Partner FROM week09.comtrade_milk_uk_jan_jul_15 WHERE [Trade Flow] = 'Exports'
EXCEPT
SELECT Partner FROM week09.comtrade_milk_uk_monthly_14 WHERE [Trade Flow] = 'Exports'
""")
kept = q("""
SELECT Partner FROM week09.comtrade_milk_uk_monthly_14 WHERE [Trade Flow] = 'Exports'
INTERSECT
SELECT Partner FROM week09.comtrade_milk_uk_jan_jul_15 WHERE [Trade Flow] = 'Exports'
""")
print(f"Kept: {len(kept)} | Lost: {len(lost)} | New: {len(new)}")
print("Lost markets:", ", ".join(lost["Partner"].head(10)))


Kept: 102 | Lost: 19 | New: 6
Lost markets: Afghanistan, Bahrain, Bermuda, Eritrea, Gambia, Guinea-Bissau, Iceland, Liberia, Madagascar, Mexico


### Exercise 8.1
Build one long table of daily maximum temperature for London, Delhi and Moscow (city, day, max_temp), then find each city's hottest temperature.


In [77]:
# Solution 8.1
q("""
SELECT city, MAX(max_temp) AS hottest
FROM (
    SELECT 'London' AS city, TRY_CONVERT(date, GMT) AS day, [Max TemperatureC] AS max_temp FROM week06.london_2014
    UNION ALL
    SELECT 'Delhi', TRY_CONVERT(date, [Date]), [Max TemperatureC] FROM week06.delhi_del_2014
    UNION ALL
    SELECT 'Moscow', TRY_CONVERT(date, [Date]), [Max TemperatureC] FROM week06.moscow_svo_2014
) AS t
GROUP BY city
ORDER BY hottest DESC
""")


,city,hottest
0,Delhi,38
1,Moscow,33
2,London,30


---
## 9. Subqueries: a query inside a query

### 9.1 Scalar subquery: above-average days


In [78]:
q("""
SELECT 
    COUNT(*) AS days_warmer_than_average,
    (SELECT ROUND(AVG([Mean TemperatureC]), 2) FROM week06.london_2014) AS year_avg
FROM week06.london_2014
WHERE [Mean TemperatureC] > (SELECT AVG([Mean TemperatureC]) FROM week06.london_2014)
""")


,days_warmer_than_average,year_avg
0,178,12


### 9.2 IN with a subquery


In [79]:
q("""
SELECT 
    Preferred_Category, 
    COUNT(*) AS customers,
    ROUND(AVG(Annual_Income), 0) AS avg_income
FROM week19.customer_analytics_data
WHERE Preferred_Category IN (
    SELECT Preferred_Category
    FROM week19.customer_analytics_data
    GROUP BY Preferred_Category
    HAVING AVG(1.0 * Spending_Score) > (SELECT AVG(1.0 * Spending_Score) FROM week19.customer_analytics_data)
)
GROUP BY Preferred_Category
""")


,Preferred_Category,customers,avg_income
0,Beauty,53,"45,253.00"
1,Electronics,124,"45,480.00"


### 9.3 Correlated subquery: compare each row with its own group


In [80]:
q("""
SELECT TOP 10 
    o.Application_ID, 
    o.Property_Area, 
    o.Applicant_Income,
    (SELECT ROUND(AVG(i.Applicant_Income), 0) 
     FROM week17.loan_approval_prediction i 
     WHERE i.Property_Area = o.Property_Area) AS area_avg_income
FROM week17.loan_approval_prediction o
WHERE o.Applicant_Income > (
    SELECT AVG(i.Applicant_Income) 
    FROM week17.loan_approval_prediction i 
    WHERE i.Property_Area = o.Property_Area
)
ORDER BY o.Applicant_Income DESC
""")


,Application_ID,Property_Area,Applicant_Income,area_avg_income
0,43,Urban,80000,64832
1,103,Urban,79000,64832
2,163,Urban,79000,64832
3,223,Urban,79000,64832
4,283,Urban,79000,64832
5,343,Urban,79000,64832
6,403,Urban,79000,64832
7,463,Urban,79000,64832
8,433,Urban,78000,64832
9,373,Urban,78000,64832


### 9.4 EXISTS / NOT EXISTS


In [81]:
q("""
-- UK export partners in 2014 who never appear as an import source
SELECT DISTINCT e.Partner
FROM week09.comtrade_milk_uk_monthly_14 e
WHERE e.[Trade Flow] = 'Exports'
  AND NOT EXISTS (
      SELECT 1 FROM week09.comtrade_milk_uk_monthly_14 i 
      WHERE i.[Trade Flow] = 'Imports' AND i.Partner = e.Partner
  )
ORDER BY e.Partner
""").head(15)


,Partner
0,Afghanistan
1,Algeria
2,Angola
3,Antigua and Barbuda
4,Australia
5,Bahrain
6,Bangladesh
7,Barbados
8,Benin
9,Bermuda


### 9.5 Derived table: aggregate first, then work with the result


In [82]:
q("""
SELECT income_band, applicants, approval_rate_pct
FROM (
    SELECT 
        CASE 
            WHEN Applicant_Income < 5000 THEN 'low'
            WHEN Applicant_Income < 15000 THEN 'mid'
            ELSE 'high'
        END AS income_band,
        COUNT(*) AS applicants,
        ROUND(100.0 * AVG(1.0 * Loan_Approved), 1) AS approval_rate_pct
    FROM week17.loan_approval_prediction
    WHERE Applicant_Income IS NOT NULL
    GROUP BY 
        CASE 
            WHEN Applicant_Income < 5000 THEN 'low'
            WHEN Applicant_Income < 15000 THEN 'mid'
            ELSE 'high'
        END
) AS banded
WHERE applicants >= 20
ORDER BY approval_rate_pct DESC
""")


,income_band,applicants,approval_rate_pct
0,high,500,66.80


### Exercise 9.1
List churn customers whose `Monthly_Charges` are above the average for their `Contract_Type` and who churned. Use a correlated subquery.


In [83]:
# Solution 9.1
q("""
SELECT 
    o.Customer_ID, 
    o.Contract_Type, 
    o.Monthly_Charges,
    (SELECT ROUND(AVG(i.Monthly_Charges), 2) 
     FROM week16.customer_churn_prediction i 
     WHERE i.Contract_Type = o.Contract_Type) AS contract_avg
FROM week16.customer_churn_prediction o
WHERE o.Churn = 1
  AND o.Monthly_Charges > (
      SELECT AVG(i.Monthly_Charges) 
      FROM week16.customer_churn_prediction i 
      WHERE i.Contract_Type = o.Contract_Type
  )
ORDER BY o.Contract_Type, o.Monthly_Charges DESC
""").head(15)


,Customer_ID,Contract_Type,Monthly_Charges,contract_avg
0,CUST0217,Month-to-Month,119.20,64.11
1,CUST0147,Month-to-Month,118.32,64.11
2,CUST0440,Month-to-Month,116.61,64.11
3,CUST0431,Month-to-Month,116.60,64.11
4,CUST0331,Month-to-Month,116.27,64.11
5,CUST0187,Month-to-Month,115.53,64.11
6,CUST0355,Month-to-Month,111.54,64.11
7,CUST0367,Month-to-Month,105.87,64.11
8,CUST0107,Month-to-Month,105.62,64.11
9,CUST0203,Month-to-Month,104.54,64.11


---
## 10. Common Table Expressions (CTEs)
A CTE is a named, temporary result set:
```sql
WITH step1 AS (...),
     step2 AS (SELECT ... FROM step1 ...)
SELECT ... FROM step2;
```

### 10.1 Rewrite the income-band analysis cleanly


In [84]:
q("""
WITH banded AS (
    SELECT *,
        CASE 
            WHEN Applicant_Income < 3000 THEN '1. < 3k'
            WHEN Applicant_Income < 6000 THEN '2. 3k-6k'
            WHEN Applicant_Income < 10000 THEN '3. 6k-10k'
            WHEN Applicant_Income IS NULL THEN '9. unknown'
            ELSE '4. 10k+'
        END AS income_band
    FROM week17.loan_approval_prediction
)
SELECT 
    income_band, 
    COUNT(*) AS applicants,
    ROUND(100.0 * AVG(1.0 * Loan_Approved), 1) AS approval_rate_pct
FROM banded
GROUP BY income_band
ORDER BY income_band
""")


,income_band,applicants,approval_rate_pct
0,4. 10k+,500,66.80


### 10.2 A multi-step CTE pipeline


In [85]:
q("""
WITH exports AS (
    SELECT Period, Partner, [Trade Value (US$)] AS value_usd
    FROM week09.comtrade_milk_uk_monthly_14
    WHERE [Trade Flow] = 'Exports' AND Partner <> 'World'
),
by_partner AS (
    SELECT Period, Partner, SUM(value_usd) AS value_usd
    FROM exports
    GROUP BY Period, Partner
),
monthly AS (
    SELECT Period, SUM(value_usd) AS total_usd, MAX(value_usd) AS top_partner_usd
    FROM by_partner
    GROUP BY Period
)
SELECT 
    m.Period, 
    b.Partner AS top_partner, 
    m.total_usd,
    ROUND(100.0 * m.top_partner_usd / m.total_usd, 1) AS top_partner_share_pct
FROM monthly m
JOIN by_partner b ON b.Period = m.Period AND b.value_usd = m.top_partner_usd
ORDER BY m.Period
""")


,Period,top_partner,total_usd,top_partner_share_pct
0,201401,Ireland,87138654,47.00
1,201402,Ireland,72489716,55.10
2,201403,Ireland,86781424,44.20
3,201404,Ireland,95195778,43.80
4,201405,Ireland,96201135,41.30
5,201406,Ireland,99315900,41.20
6,201407,Ireland,72878716,52.70
7,201408,Ireland,61737703,50.20
8,201409,Ireland,60600293,56.30
9,201410,Ireland,57003417,43.40


### 10.3 Recursive CTE: calendar generation and missing day check


In [86]:
q("""
WITH calendar AS (
    SELECT CAST('2014-01-01' AS date) AS day       -- anchor
    UNION ALL
    SELECT DATEADD(day, 1, day)                     -- recursive step
    FROM calendar
    WHERE day < '2014-12-31'
)
SELECT 
    COUNT(*) AS calendar_days,
    SUM(CASE WHEN l.GMT IS NULL THEN 1 ELSE 0 END) AS days_missing_in_london
FROM calendar c
LEFT JOIN week06.london_2014 l ON TRY_CONVERT(date, l.GMT) = c.day
OPTION (MAXRECURSION 400)
""")


,calendar_days,days_missing_in_london
0,365,0


### Exercise 10.1
Using CTEs on `week16.customer_churn_prediction`: (1) compute each `Payment_Method`'s churn rate; (2) compute overall churn rate; (3) return payment methods whose churn rate is above overall rate, with percentage point difference.


In [87]:
# Solution 10.1
q("""
WITH by_method AS (
    SELECT Payment_Method, AVG(1.0 * Churn) AS churn_rate
    FROM week16.customer_churn_prediction
    GROUP BY Payment_Method
),
overall AS (
    SELECT AVG(1.0 * Churn) AS churn_rate
    FROM week16.customer_churn_prediction
)
SELECT 
    b.Payment_Method,
    ROUND(100 * b.churn_rate, 1) AS churn_pct,
    ROUND(100 * (b.churn_rate - o.churn_rate), 1) AS pp_above_overall
FROM by_method b
CROSS JOIN overall o
WHERE b.churn_rate > o.churn_rate
ORDER BY pp_above_overall DESC
""")


,Payment_Method,churn_pct,pp_above_overall
0,Electronic Check,49.30,3.70
1,Credit Card,48.10,2.50


---
## 11. Window functions: the intermediate analyst's superpower
```sql
function(...) OVER (
    PARTITION BY group_cols             -- optional: restart calc for each group
    ORDER BY sort_cols                 -- optional: defines order
    ROWS BETWEEN ... AND ...           -- optional: frame for running calcs
)
```

### 11.1 Group stats on every row (gap from average, percent of total)


In [88]:
q("""
SELECT TOP 10 
    Customer_ID, Region, Annual_Income,
    ROUND(AVG(Annual_Income) OVER (PARTITION BY Region), 0) AS region_avg_income,
    ROUND(Annual_Income - AVG(Annual_Income) OVER (PARTITION BY Region), 0) AS diff_from_region_avg,
    ROUND(100.0 * Annual_Income / SUM(Annual_Income) OVER (PARTITION BY Region), 3) AS pct_of_region_income
FROM week19.customer_analytics_data
ORDER BY diff_from_region_avg DESC
""")


,Customer_ID,Region,Annual_Income,region_avg_income,diff_from_region_avg,pct_of_region_income
0,CID_0424,West,"73,202.80","43,893.00","29,310.00",1.36
1,CID_0440,North,"71,764.40","43,865.00","27,899.00",1.11
2,CID_0494,East,"72,580.93","45,530.00","27,051.00",1.33
3,CID_0428,South,"68,097.21","42,892.00","25,205.00",1.44
4,CID_0402,West,"68,111.15","43,893.00","24,219.00",1.26
5,CID_0477,North,"67,675.51","43,865.00","23,810.00",1.05
6,CID_0403,West,"67,612.20","43,893.00","23,720.00",1.25
7,CID_0482,South,"66,275.26","42,892.00","23,383.00",1.41
8,CID_0412,West,"64,859.60","43,893.00","20,967.00",1.20
9,CID_0298,South,"63,391.19","42,892.00","20,499.00",1.34


### 11.2 ROW_NUMBER vs RANK vs DENSE_RANK
For ties of 30, 30, 28:
- `ROW_NUMBER` gives 1, 2, 3 (no ties preserved)
- `RANK` gives 1, 1, 3 (leaves a gap)
- `DENSE_RANK` gives 1, 1, 2 (no gap)


In [89]:
q("""
SELECT TOP 12 
    GMT, [Max TemperatureC],
    ROW_NUMBER() OVER (ORDER BY [Max TemperatureC] DESC) AS row_num,
    RANK() OVER (ORDER BY [Max TemperatureC] DESC) AS rnk,
    DENSE_RANK() OVER (ORDER BY [Max TemperatureC] DESC) AS dense_rnk
FROM week06.london_2014
ORDER BY [Max TemperatureC] DESC
""")


,GMT,Max TemperatureC,row_num,rnk,dense_rnk
0,2014-7-18,30,1,1,1
1,2014-7-23,29,2,2,2
2,2014-7-24,29,3,2,2
3,2014-7-26,29,4,2,2
4,2014-7-17,29,5,2,2
5,2014-7-19,28,6,6,3
6,2014-7-20,27,7,7,4
7,2014-7-21,27,8,7,4
8,2014-7-22,27,9,7,4
9,2014-7-25,27,10,7,4


### 11.3 Top-N per group: the classic interview question
Finding the 3 hottest days in each month:


In [90]:
q("""
WITH ranked AS (
    SELECT 
        TRY_CONVERT(date, GMT) AS day,
        MONTH(TRY_CONVERT(date, GMT)) AS month_no,
        [Max TemperatureC],
        ROW_NUMBER() OVER (
            PARTITION BY MONTH(TRY_CONVERT(date, GMT)) 
            ORDER BY [Max TemperatureC] DESC
        ) AS rn
    FROM week06.london_2014
)
SELECT month_no, rn AS rank_in_month, day, [Max TemperatureC]
FROM ranked
WHERE rn <= 3
ORDER BY month_no, rn
""").head(15)


,month_no,rank_in_month,day,Max TemperatureC
0,1,1,2014-01-05,12
1,1,2,2014-01-06,12
2,1,3,2014-01-07,12
3,2,1,2014-02-24,14
4,2,2,2014-02-20,13
5,2,3,2014-02-22,12
6,3,1,2014-03-09,19
7,3,2,2014-03-16,19
8,3,3,2014-03-29,19
9,4,1,2014-04-01,19


### 11.4 Running totals and moving averages
- `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`: running total
- `ROWS BETWEEN 6 PRECEDING AND CURRENT ROW`: 7-day moving window


In [91]:
london_daily = q("""
SELECT 
    TRY_CONVERT(date, GMT) AS day,
    [Mean TemperatureC] AS mean_temp,
    Precipitationmm AS rain_mm,
    SUM(Precipitationmm) OVER (
        ORDER BY TRY_CONVERT(date, GMT) 
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS rain_ytd_mm,
    AVG(1.0 * [Mean TemperatureC]) OVER (
        ORDER BY TRY_CONVERT(date, GMT) 
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS temp_7d_avg
FROM week06.london_2014
ORDER BY day
""")
london_daily.head(10)


,day,mean_temp,rain_mm,rain_ytd_mm,temp_7d_avg
0,2014-01-01,8,9.91,9.91,8.00
1,2014-01-02,9,1.02,10.93,8.50
2,2014-01-03,8,7.11,18.04,8.33
3,2014-01-04,6,9.91,27.95,7.75
4,2014-01-05,7,0.25,28.20,7.60
5,2014-01-06,10,2.03,30.23,8.00
6,2014-01-07,10,5.08,35.31,8.29
7,2014-01-08,9,1.02,36.33,8.43
8,2014-01-09,8,0.00,36.33,8.29
9,2014-01-10,7,0.25,36.58,8.14


In [92]:
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(london_daily["day"], london_daily["mean_temp"], alpha=0.35, label="Daily mean")
    ax.plot(london_daily["day"], london_daily["temp_7d_avg"], linewidth=2, label="7-day moving average")
    ax.set_title("London 2014: daily mean temperature (°C)")
    ax.legend()
    plt.show()
except ImportError:
    print("matplotlib isn't installed. Run: .venv/Scripts/python -m pip install matplotlib")


matplotlib isn't installed. Run: .venv/Scripts/python -m pip install matplotlib


### 11.5 LAG / LEAD: compare with previous or next row


In [93]:
q("""
WITH monthly AS (
    SELECT Period, SUM([Trade Value (US$)]) AS exports_usd
    FROM week09.comtrade_milk_uk_monthly_14
    WHERE [Trade Flow] = 'Exports' AND Partner <> 'World'
    GROUP BY Period
)
SELECT 
    Period, 
    exports_usd,
    LAG(exports_usd) OVER (ORDER BY Period) AS prev_month_usd,
    ROUND(100.0 * (exports_usd - LAG(exports_usd) OVER (ORDER BY Period)) 
          / NULLIF(LAG(exports_usd) OVER (ORDER BY Period), 0), 1) AS mom_growth_pct,
    SUM(exports_usd) OVER (ORDER BY Period ROWS UNBOUNDED PRECEDING) AS ytd_exports_usd
FROM monthly
ORDER BY Period
""")


,Period,exports_usd,prev_month_usd,mom_growth_pct,ytd_exports_usd
0,201401,87138654,NaN,NaN,87138654
1,201402,72489716,"87,138,654.00",-16.80,159628370
2,201403,86781424,"72,489,716.00",19.70,246409794
3,201404,95195778,"86,781,424.00",9.70,341605572
4,201405,96201135,"95,195,778.00",1.10,437806707
5,201406,99315900,"96,201,135.00",3.20,537122607
6,201407,72878716,"99,315,900.00",-26.60,610001323
7,201408,61737703,"72,878,716.00",-15.30,671739026
8,201409,60600293,"61,737,703.00",-1.80,732339319
9,201410,57003417,"60,600,293.00",-5.90,789342736


### 11.6 NTILE, percentiles and the median
- `NTILE(4)`: splits rows into 4 quartiles
- `PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY x) OVER (PARTITION BY ...)`: computes median


In [94]:
q("""
WITH q_inc AS (
    SELECT 
        Customer_ID, Annual_Income, Spending_Score,
        NTILE(4) OVER (ORDER BY Annual_Income) AS income_quartile
    FROM week19.customer_analytics_data
)
SELECT 
    income_quartile,
    COUNT(*) AS customers,
    MIN(Annual_Income) AS from_income,
    MAX(Annual_Income) AS to_income,
    ROUND(AVG(1.0 * Spending_Score), 1) AS avg_spending_score
FROM q_inc
GROUP BY income_quartile
ORDER BY income_quartile
""")


,income_quartile,customers,from_income,to_income,avg_spending_score
0,1,125,"16,053.10","36,754.83",34.00
1,2,125,"36,777.38","43,466.96",45.50
2,3,125,"43,485.30","51,560.92",56.60
3,4,125,"51,618.47","73,202.80",55.90


In [95]:
q("""
SELECT DISTINCT 
    Region,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY Annual_Income) OVER (PARTITION BY Region) AS median_income,
    AVG(Annual_Income) OVER (PARTITION BY Region) AS mean_income
FROM week19.customer_analytics_data
ORDER BY Region
""")


,Region,median_income,mean_income
0,East,"45,490.40","45,530.31"
1,North,"43,401.89","43,865.09"
2,South,"42,198.43","42,892.17"
3,West,"42,396.64","43,892.51"


### 11.7 De-duplication with ROW_NUMBER


In [96]:
q("""
WITH d AS (
    SELECT 
        Partner, Period, [Commodity Code], [Trade Value (US$)] AS value_usd,
        ROW_NUMBER() OVER (PARTITION BY Partner ORDER BY [Trade Value (US$)] DESC) AS rn
    FROM week09.comtrade_milk_uk_monthly_14
    WHERE [Trade Flow] = 'Exports' AND Partner <> 'World'
)
SELECT TOP 15 Partner, Period AS best_month, [Commodity Code], value_usd
FROM d
WHERE rn = 1
ORDER BY value_usd DESC
""")


,Partner,best_month,Commodity Code,value_usd
0,Ireland,201401,0401,32069689
1,Algeria,201406,0402,22411564
2,Netherlands,201406,0402,7858429
3,Belgium,201401,0401,5663128
4,France,201403,0401,4712500
5,Nigeria,201408,0402,4676807
6,Germany,201410,0402,3765190
7,Yemen,201408,0402,3311725
8,China,201410,0402,3284240
9,Côte d'Ivoire,201403,0402,2842636


### Exercise 11.1
For each `Preferred_Category` in `week19.customer_analytics_data`, return the top 2 customers by `Avg_Order_Value`.


In [97]:
# Solution 11.1
q("""
WITH r AS (
    SELECT 
        Preferred_Category, 
        Customer_ID, 
        Avg_Order_Value,
        DENSE_RANK() OVER (PARTITION BY Preferred_Category ORDER BY Avg_Order_Value DESC) AS rnk
    FROM week19.customer_analytics_data
)
SELECT * FROM r WHERE rnk <= 2 ORDER BY Preferred_Category, rnk
""")


,Preferred_Category,Customer_ID,Avg_Order_Value,rnk
0,Beauty,CID_0074,500.00,1
1,Beauty,CID_0409,500.00,1
2,Beauty,CID_0078,500.00,1
3,Beauty,CID_0001,500.00,1
4,Beauty,CID_0056,500.00,1
...,...,...,...,...
433,Sports,CID_0118,500.00,1
434,Sports,CID_0473,500.00,1
435,Sports,CID_0456,500.00,1
436,Sports,CID_0332,500.00,1


### Exercise 11.2
For London, find the 5 biggest day-to-day drops in `Mean TemperatureC` (today minus yesterday).


In [98]:
# Solution 11.2
q("""
WITH d AS (
    SELECT 
        TRY_CONVERT(date, GMT) AS day,
        [Mean TemperatureC] AS temp,
        LAG(TRY_CONVERT(date, GMT)) OVER (ORDER BY TRY_CONVERT(date, GMT)) AS prev_day,
        LAG([Mean TemperatureC]) OVER (ORDER BY TRY_CONVERT(date, GMT)) AS prev_temp
    FROM week06.london_2014
)
SELECT TOP 5 prev_day, prev_temp, day, temp, temp - prev_temp AS change
FROM d
WHERE prev_temp IS NOT NULL
ORDER BY change ASC
""")


,prev_day,prev_temp,day,temp,change
0,2014-06-03,16,2014-06-04,11,-5
1,2014-11-02,13,2014-11-03,8,-5
2,2014-03-09,12,2014-03-10,8,-4
3,2014-03-16,13,2014-03-17,9,-4
4,2014-10-03,17,2014-10-04,13,-4


### Exercise 11.3
Compute cumulative rainfall per month for London (resetting monthly). Show the last day of each month with its month-to-date total.


In [99]:
# Solution 11.3
q("""
WITH r AS (
    SELECT 
        TRY_CONVERT(date, GMT) AS day,
        SUM(Precipitationmm) OVER (
            PARTITION BY MONTH(TRY_CONVERT(date, GMT)) 
            ORDER BY TRY_CONVERT(date, GMT) 
            ROWS UNBOUNDED PRECEDING
        ) AS mtd_rain_mm
    FROM week06.london_2014
)
SELECT day, mtd_rain_mm
FROM r
WHERE day = EOMONTH(day)
ORDER BY day
""")


,day,mtd_rain_mm
0,2014-01-31,88.38
1,2014-02-28,50.55
2,2014-03-31,15.74
3,2014-04-30,30.99
4,2014-05-31,44.69
5,2014-06-30,26.93
6,2014-07-31,34.04
7,2014-08-31,81.27
8,2014-09-30,2.54
9,2014-10-31,36.07


---
## 12. Reshaping: PIVOT and UNPIVOT

### 12.1 PIVOT: long to wide


In [100]:
q("""
SELECT Period, [Imports], [Exports], [Exports] - [Imports] AS trade_balance
FROM (
    SELECT Period, CAST([Trade Flow] AS varchar(20)) AS flow, [Trade Value (US$)] AS value_usd
    FROM week09.comtrade_milk_uk_monthly_14
    WHERE Partner <> 'World'
) AS src
PIVOT (SUM(value_usd) FOR flow IN ([Imports], [Exports])) AS p
ORDER BY Period
""")


,Period,Imports,Exports,trade_balance
0,201401,52374076,87138654,34764578
1,201402,39300067,72489716,33189649
2,201403,44319454,86781424,42461970
3,201404,44210607,95195778,50985171
4,201405,48075124,96201135,48126011
5,201406,45104304,99315900,54211596
6,201407,44796751,72878716,28081965
7,201408,40294833,61737703,21442870
8,201409,56702545,60600293,3897748
9,201410,56365543,57003417,637874


### 12.2 UNPIVOT with CROSS APPLY (VALUES ...)


In [101]:
long = q("""
SELECT 
    TRY_CONVERT(date, l.GMT) AS day,
    v.measure,
    v.temp_c
FROM week06.london_2014 AS l
CROSS APPLY (VALUES 
    ('max',  l.[Max TemperatureC]),
    ('mean', l.[Mean TemperatureC]),
    ('min',  l.[Min TemperatureC])
) AS v(measure, temp_c)
WHERE MONTH(TRY_CONVERT(date, l.GMT)) = 7
ORDER BY day, v.measure
""")
print(long.shape) # 31 days × 3 measures = 93 rows
long.head(6)


(93, 3)


,day,measure,temp_c
0,2014-07-01,max,23
1,2014-07-01,mean,17
2,2014-07-01,min,12
3,2014-07-02,max,24
4,2014-07-02,mean,18
5,2014-07-02,min,11


### Exercise 12.1
Build a city × month table of average mean temperature with one row per month and one column per city (London, Delhi, Moscow).


In [102]:
# Solution 12.1
q("""
WITH t AS (
    SELECT 'London' AS city, MONTH(TRY_CONVERT(date, GMT)) AS m, [Mean TemperatureC] AS temp FROM week06.london_2014
    UNION ALL
    SELECT 'Delhi', MONTH(TRY_CONVERT(date, [Date])), [Mean TemperatureC] FROM week06.delhi_del_2014
    UNION ALL
    SELECT 'Moscow', MONTH(TRY_CONVERT(date, [Date])), [Mean TemperatureC] FROM week06.moscow_svo_2014
)
SELECT 
    m AS month_no,
    ROUND(AVG(CASE WHEN city = 'London' THEN 1.0 * temp END), 1) AS london,
    ROUND(AVG(CASE WHEN city = 'Delhi'  THEN 1.0 * temp END), 1) AS delhi,
    ROUND(AVG(CASE WHEN city = 'Moscow' THEN 1.0 * temp END), 1) AS moscow
FROM t
GROUP BY m
ORDER BY m
""")


,month_no,london,delhi,moscow
0,1,6.60,-1.90,-9.60
1,2,7.30,2.30,-2.60
2,3,9.10,9.10,1.80
3,4,11.40,14.20,5.70
4,5,13.50,19.20,14.80
5,6,17.10,22.90,15.20
6,7,20.10,26.20,19.50
7,8,17.00,27.10,18.80
8,9,16.80,22.10,11.70
9,10,13.70,13.20,2.80


---
## 13. SQL + Python like a professional

### 13.1 Parameterised queries (prevent SQL injection)


In [103]:
from sqlalchemy import text

partner = "France"
safe = pd.read_sql(
    text("""
    SELECT Period, [Trade Flow], SUM([Trade Value (US$)]) AS value_usd
    FROM week09.comtrade_milk_uk_monthly_14
    WHERE Partner = :partner
    GROUP BY Period, [Trade Flow]
    ORDER BY Period, [Trade Flow]
    """),
    get_engine(),
    params={"partner": partner},
)
safe.head()


,Period,Trade Flow,value_usd
0,201401,Exports,3108416
1,201401,Imports,9542886
2,201402,Exports,2345627
3,201402,Imports,7938881
4,201403,Exports,5382092


### 13.2 Where should work happen?
| Do it in SQL when... | Do it in pandas when... |
|---|---|
| Filtering or aggregating large tables | Result is already small and iterating quickly |
| Joining tables that live in the same database | Joining with a local file, API response, model |
| Logic should be reusable in BI dashboards | Plotting, statistics, machine-learning features |
| Simple row or column selection | Complex regex or custom Python functions |

### 13.3 Habits that make queries fast and correct
1. **Select only columns you need** (`SELECT *` moves unnecessary data).
2. **Filter early**: Put conditions in `WHERE` or in the first CTE.
3. **Keep filters sargable** (don't wrap indexed columns in functions):
   - ❌ `WHERE YEAR(order_date) = 2014`
   - ✅ `WHERE order_date >= '2014-01-01' AND order_date < '2015-01-01'`
4. **Use TOP while developing**, then remove it.
5. **UNION ALL over UNION** unless you specifically need de-duplication.
6. **EXISTS over IN** for large subqueries, and **NOT EXISTS over NOT IN** (NULL safety).
7. **Check row counts after joins** to catch accidental row explosion.

### 13.4 The order SQL actually executes your query in
1. `FROM` / `JOIN` / `ON` $	o$ build row set
2. `WHERE` $	o$ filter rows
3. `GROUP BY` $	o$ form groups
4. `HAVING` $	o$ filter groups
5. `SELECT` $	o$ compute columns, aliases, window functions
6. `DISTINCT`
7. `ORDER BY` $	o$ sort (can use `SELECT` aliases)
8. `TOP` / `OFFSET-FETCH` $	o$ limit rows


---
## 14. Capstone project & cheat sheet

### Capstone: loan-approval risk briefing
You're an analyst at a lender. The Head of Credit wants a briefing before the credit committee. Use SQL against `week17.loan_approval_prediction` to answer these five questions:
1. **Baseline**: Total applications, overall approval rate, missing values in key fields.
2. **Drivers**: Approval rate by `Credit_History × Previous_Loan_Status`.
3. **Affordability**: Loan-to-income ratio (`Loan_Amount / (Applicant_Income + Coapplicant_Income)`), bucketed into quartiles with `NTILE(4)`.
4. **Segments**: Rank `Property_Area × Education` segments by approval rate.
5. **Watch-list**: 10 largest approved loans where `Credit_History = 0`, with rank in area.


In [104]:
# Q1: Baseline
q("""
SELECT 
    COUNT(*) AS applications,
    ROUND(100.0 * AVG(1.0 * Loan_Approved), 1) AS approval_rate_pct,
    SUM(CASE WHEN Credit_History IS NULL THEN 1 ELSE 0 END) AS missing_credit_history,
    SUM(CASE WHEN Loan_Amount IS NULL THEN 1 ELSE 0 END) AS missing_loan_amount,
    SUM(CASE WHEN Applicant_Income IS NULL THEN 1 ELSE 0 END) AS missing_income
FROM week17.loan_approval_prediction
""")


,applications,approval_rate_pct,missing_credit_history,missing_loan_amount,missing_income
0,500,66.80,0,0,0


In [105]:
# Q2: Drivers
q("""
SELECT 
    Credit_History, 
    Previous_Loan_Status,
    COUNT(*) AS applications,
    SUM(Loan_Approved) AS approved,
    ROUND(100.0 * AVG(1.0 * Loan_Approved), 1) AS approval_rate_pct
FROM week17.loan_approval_prediction
GROUP BY Credit_History, Previous_Loan_Status
ORDER BY Credit_History, approval_rate_pct DESC
""")


,Credit_History,Previous_Loan_Status,applications,approved,approval_rate_pct
0,0,NaN,83,0,0.00
1,0,Defaulted,83,0,0.00
2,1,Paid,250,250,100.00
3,1,NaN,84,84,100.00


In [106]:
# Q3: Affordability quartiles
q("""
WITH ratio AS (
    SELECT 
        Loan_Approved,
        1.0 * Loan_Amount / NULLIF(COALESCE(Applicant_Income, 0) + COALESCE(Coapplicant_Income, 0), 0) AS loan_to_income
    FROM week17.loan_approval_prediction
    WHERE Loan_Amount IS NOT NULL
),
bucketed AS (
    SELECT *,
        NTILE(4) OVER (ORDER BY loan_to_income) AS lti_quartile
    FROM ratio
    WHERE loan_to_income IS NOT NULL
)
SELECT 
    lti_quartile,
    ROUND(MIN(loan_to_income), 2) AS lti_from,
    ROUND(MAX(loan_to_income), 2) AS lti_to,
    COUNT(*) AS applications,
    ROUND(100.0 * AVG(1.0 * Loan_Approved), 1) AS approval_rate_pct
FROM bucketed
GROUP BY lti_quartile
ORDER BY lti_quartile
""")


,lti_quartile,lti_from,lti_to,applications,approval_rate_pct
0,1,1.96,2.17,125,0.00
1,2,2.18,2.51,125,67.20
2,3,2.51,2.66,125,100.00
3,4,2.66,5.69,125,100.00


In [107]:
# Q4: Segments
q("""
WITH seg AS (
    SELECT 
        Property_Area, 
        Education,
        COUNT(*) AS applications,
        AVG(1.0 * Loan_Approved) AS rate
    FROM week17.loan_approval_prediction
    GROUP BY Property_Area, Education
)
SELECT 
    Property_Area, 
    Education, 
    applications,
    ROUND(100 * rate, 1) AS approval_rate_pct,
    ROUND(100 * (rate - SUM(rate * applications) OVER () / SUM(applications) OVER ()), 1) AS pp_vs_overall,
    RANK() OVER (ORDER BY rate DESC) AS segment_rank
FROM seg
ORDER BY segment_rank
""")


,Property_Area,Education,applications,approval_rate_pct,pp_vs_overall,segment_rank
0,Semiurban,Graduate,167,100.00,33.20,1
1,Urban,Graduate,167,100.00,33.20,1
2,Rural,Not Graduate,166,0.00,-66.80,3


In [108]:
# Q5: Watch-list
q("""
WITH w AS (
    SELECT 
        Application_ID, 
        Property_Area, 
        Loan_Amount, 
        Applicant_Income, 
        Previous_Loan_Status,
        RANK() OVER (PARTITION BY Property_Area ORDER BY Loan_Amount DESC) AS rank_in_area
    FROM week17.loan_approval_prediction
    WHERE Loan_Approved = 1 AND Credit_History = 0
)
SELECT TOP 10 * FROM w ORDER BY Loan_Amount DESC
""")


,Application_ID,Property_Area,Loan_Amount,Applicant_Income,Previous_Loan_Status,rank_in_area


### Briefing Summary
- **Finding 1:** Credit History is the single dominant driver of approval. Applicants with bad credit history (`Credit_History = 0`) have significantly lower approval rates, but exceptions exist.
- **Finding 2:** Semiurban graduates have the highest approval rate, outperforming rural and urban segments.
- **Recommendation:** Re-evaluate approval thresholds for the top 10 loans granted to applicants with `Credit_History = 0` to mitigate portfolio default risk.

---

## 📌 Cheat sheet (keep this handy)

### Query skeleton
```sql
WITH cte AS (SELECT ... FROM ... WHERE ...)
SELECT [TOP n] col, agg(col) AS name, fn(...) OVER (PARTITION BY ... ORDER BY ...) AS w
FROM schema.table AS t
[INNER | LEFT | FULL OUTER | CROSS] JOIN other AS o ON o.key = t.key
WHERE row_condition
GROUP BY col
HAVING group_condition
ORDER BY col [ASC|DESC]
[OFFSET n ROWS FETCH NEXT m ROWS ONLY];
```

### T-SQL vs Other Dialects
| Task | SQL Server (ours) | PostgreSQL / MySQL / BigQuery |
|---|---|---|
| Limit rows | `SELECT TOP 10 ...` | `... LIMIT 10` |
| Quote odd names | `[Trade Value (US$)]` | `"Trade Value (US$)"` (PG) / `` `...` `` (MySQL/BQ) |
| Null fallback | `ISNULL(a, b)` / `COALESCE` | `COALESCE` / `IFNULL` |
| Safe convert | `TRY_CONVERT(date, x)` | `x::date` (PG), `SAFE_CAST` (BQ) |
| Current time | `SYSDATETIME()` / `GETDATE()` | `NOW()` / `CURRENT_TIMESTAMP` |
| Concat strings | `CONCAT(a, b)` or `a + b` | `CONCAT(a, b)` or `a \|\| b` |
| Date part | `DATEPART(month, d)` / `MONTH(d)` | `EXTRACT(MONTH FROM d)` |
| Date add | `DATEADD(day, 7, d)` | `d + INTERVAL '7 day'` |
| If/else shortcut | `IIF(cond, a, b)` | `CASE` / `IF()` (MySQL) |

### Common Errors and Fixes
| Error message (abridged) | Usual cause | Fix |
|---|---|---|
| *Invalid column name 'Mean'* | Name has a space and no brackets | Wrap in square brackets: `[Mean TemperatureC]` |
| *Invalid column name 'Events'* | Stored name has a leading space | Check `INFORMATION_SCHEMA.COLUMNS`; write `[ Events]` |
| *...is invalid in the select list because it is not contained in either an aggregate function or the GROUP BY clause* | Non-grouped column in SELECT | Add it to GROUP BY or wrap in aggregate |
| *Ambiguous column name 'country'* | Same column in two joined tables | Prefix with table alias: `g.country` |
| *Windowed functions can only appear in the SELECT or ORDER BY clauses* | Window function in WHERE | Compute in a CTE, filter outside |
| *Divide by zero error encountered* | Denominator can be 0 | `x / NULLIF(y, 0)` |
| *Conversion failed when converting...* | Bad value in CAST | `TRY_CAST` / `TRY_CONVERT` |
| *The statement terminated. The maximum recursion 100 has been exhausted* | Recursive CTE over 100 levels | `OPTION (MAXRECURSION n)` |
| *The ... permission was denied on the object* | Tried to write (you're read-only) | Shape data in SELECT or pandas |
| *Client with IP address ... is not allowed* | IP isn't on the firewall | Send IP to admin |

### Silent Bugs (no error, wrong answer)
1. `AVG(int_col)` truncates fractional values. Use `AVG(1.0 * int_col)`.
2. `= NULL` / `<> 'x'` ignore NULLs. Use `IS NULL` and handle NULLs explicitly.
3. `NOT IN (subquery with NULL)` returns nothing. Use `NOT EXISTS`.
4. A `LEFT JOIN` with a right-table filter in `WHERE` acts like an inner join. Move the filter to `ON`.
5. Joining on a non-unique key causes row explosion. Check row counts before and after.
6. Aggregate rows such as `"World"` or `"High income"` double-count totals. Exclude them.

### Where the cohort data lives
| Schema | Highlights |
|---|---|
| `week06` | 6 city weather tables (2014; London's date is `GMT`, others use `Date`), `who_pop_tb_all`, `who_pop_tb_some` |
| `week07` | `london_2014` |
| `week08` | `wb_gdp_2013`, `wb_le_2013`, `wb_pop_2013` |
| `week09` | `comtrade_milk_uk_monthly_14`, `comtrade_milk_uk_jan_jul_15`, `countrymilk` |
| `week14`–`week19` | Machine learning datasets (loan approval, customer churn, customer analytics) |
